# 1. The @tool Decorator

The **easiest and most common** way to create a tool: write a normal Python function, add `@tool`
above it, and it becomes a full LangChain tool the model can call.

---

## 1. Simple Definition

> **Kid version:** You already know how to write a little Python function. Putting `@tool` on top is
> like sticking a **name tag** on it that says "Hi model, you're allowed to use me, and here's what I
> do." Now the AI can pick it up and use it.

**Professional definition:** `@tool` is a decorator that converts a Python function into a
`StructuredTool`. It automatically derives the tool's **name** (function name), **description**
(docstring), and **argument schema** (type hints) so the model knows how to call it.

```python
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

print(multiply.name)          # 'multiply'
print(multiply.description)   # 'Multiply two numbers together.'
print(multiply.args)          # {'a': {'type': 'integer', ...}, 'b': {...}}
print(multiply.invoke({"a": 3, "b": 4}))   # 12
```

---

## 2. Why Does It Exist?

**The problem:** For the model to call a function, it needs a machine-readable **schema**: the tool's
name, what it does, and its arguments with types. Writing that JSON schema by hand for every function
is tedious and easy to get out of sync with the code.

### Before (manual schema)

```python
weather_schema = {
    "name": "get_weather",
    "description": "Get weather for a city",
    "parameters": {
        "type": "object",
        "properties": {"city": {"type": "string", "description": "the city"}},
        "required": ["city"],
    },
}
# ...and you still have to wire the actual function separately. Two things to keep in sync.
```

### After (`@tool`)

```python
@tool
def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"18°C in {city}"
# name, description, and schema are ALL derived from the function automatically.
```

One source of truth (the function) → the schema is generated from your **type hints** and
**docstring**. Less code, no drift.

---

## 3. Real-Life Analogy

**Putting a label on a machine in a workshop** 🏷️. The machine already works; you just add a clear
label ("Drill — makes holes, needs: material, hole size") so anyone (the model) knows what it does and
what inputs it needs, without reading the internal wiring.

---

## 4. Where It Fits in LangChain Architecture

```
BaseTool
    │
    ▼
StructuredTool          ← what @tool actually produces
    ▲
    │ created by
  @tool  (decorator over your function)
```

- `@tool` is a **shortcut** that builds a `StructuredTool` from your function.
- The resulting object has everything the model needs: `name`, `description`, `args_schema`,
  and an `.invoke()` method.

---

## 5. Internal Working

```
  @tool
  def get_weather(city: str) -> str:
      """Get weather for a city."""
      ...
        │
        ▼
  INSPECT the function:
     - name        ← function name  → "get_weather"
     - description ← docstring       → "Get weather for a city."
     - args_schema ← type hints      → {city: string, required}
        │
        ▼
  WRAP into a StructuredTool object with an .invoke() that calls your function
        │
        ▼
  Ready to bind_tools([...]) to a model (file 5)
```

The docstring and type hints aren't just documentation — they are **sent to the model** and directly
affect how well it uses the tool.

---

## 6. Attributes / Options

### `the docstring` (→ description)

**Definition:** The function's docstring becomes the tool's `description` — the model reads this to
decide *when* to use the tool.

**Why it exists:** The description is the model's main clue about the tool's purpose.

**When developers use it:** Always — write a clear, specific docstring.

**Real-life use case:** The instruction label on the machine.

```python
@tool
def search_docs(query: str) -> str:
    """Search the company knowledge base for a query and return the top matching passage.
    Use this when the user asks about internal policies or product details."""
    ...
```

> 💡 A vague docstring = the model misuses or ignores the tool. Be specific about *what it does* and
> *when to use it*.

---

### `type hints` (→ args_schema)

**Definition:** Parameter type hints define the argument schema and types the model must provide.

**Why it exists:** The model needs to know each argument's name and type.

**Real-life use case:** The list of inputs the machine needs.

```python
@tool
def book_flight(origin: str, destination: str, passengers: int) -> str:
    """Book a flight."""
    ...
# schema: origin:str, destination:str, passengers:int  (all required)
```

---

### `name` (override)

**Definition:** By default the tool name = function name; you can override it.

**Why it exists:** Sometimes you want a cleaner/clearer tool name than the Python function name.

```python
@tool("web_search")          # custom name
def _search_impl(query: str) -> str:
    """Search the web."""
    ...
```

---

### `args_schema` (Pydantic, for richer arguments)

**Definition:** Pass a Pydantic model to precisely control argument descriptions/validation.

**Why it exists:** Per-argument descriptions dramatically improve how the model fills them.

**When developers use it:** When arguments need explanations or validation.

```python
from pydantic import BaseModel, Field

class SearchInput(BaseModel):
    query: str = Field(description="the search query, phrased as keywords")
    top_k: int = Field(default=5, description="how many results to return")

@tool(args_schema=SearchInput)
def web_search(query: str, top_k: int = 5) -> str:
    """Search the web."""
    ...
```

---

### `return_direct`

**Definition:** If `True`, an agent returns this tool's output **directly** to the user instead of
looping back to the model.

**Why it exists:** For tools whose raw output *is* the final answer (saves a round-trip and tokens).
```
Internal woprking: (return_direct=False)
User
 ↓
LLM
 ↓
multiply tool
 ↓
100
 ↓
LLM
 ↓
Final answer

The tool result goes back to the LLM, and the LLM gets a chance to explain or continue.

Internal woprking: (return_direct=True)
User
 ↓
LLM
 ↓
multiply tool
 ↓
100
 ↓
LLM
 ↓
Final answer

The agent stops after the tool and returns the tool's result directly to the user instead of sending it back through the LLM.
```

```python
@tool(return_direct=True)
def get_balance(account_id: str) -> str:
    """Return the account balance."""
    ...
```

---

### `response_format`="content_and_artifact"

**Definition:** Lets a tool return both a **string for the model** and a raw **artifact** for your code
(e.g. a dataframe, image bytes) that the model shouldn't see in full.

**Why it exists:** Some tool outputs are huge/binary; the model only needs a summary.

```python
@tool(response_format="content_and_artifact")
def run_query(sql: str) -> tuple[str, list]:
    """Run a SQL query."""
    rows = [...]
    return f"Returned {len(rows)} rows.", rows   # (content_for_model, artifact_for_code)
```

---

## `Async tools`

```python
@tool
async def fetch_url(url: str) -> str:
    """Fetch a URL asynchronously."""
    async with httpx.AsyncClient() as client:
        return (await client.get(url)).text
# Works with .ainvoke() and async agents.
```


In [1]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

# Initialize the language model
llm = ChatOllama(model="qwen3:8b")

# step 1: define the tool
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

d:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("=" * 40)
print("🛠️  Tool Information")
print("=" * 40)
print(f"Name        : {multiply.name}")
print(f"Description : {multiply.description}")
print(f"Arguments   : {multiply.args}")
print("=" * 40)

🛠️  Tool Information
Name        : multiply
Description : Multiply two numbers.
Arguments   : {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [3]:
multiply.args_schema.model_json_schema()

{'description': 'Multiply two numbers.',
 'properties': {'a': {'title': 'A', 'type': 'integer'},
  'b': {'title': 'B', 'type': 'integer'}},
 'required': ['a', 'b'],
 'title': 'multiply',
 'type': 'object'}

In [4]:
multiply.invoke({"a": 5, "b": 3})

15

In [5]:
#step2: bind the tool to the language model
llm_with_tools = llm.bind_tools([multiply])

In [6]:
# step3: tool calling
msg = llm_with_tools.invoke("What's 12 times 7?")
msg

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-08-08T14:25:29.2593329Z', 'done': True, 'done_reason': 'stop', 'total_duration': 7956339100, 'load_duration': 105752800, 'prompt_eval_count': 143, 'prompt_eval_duration': 523662000, 'eval_count': 103, 'eval_duration': 7284922000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019fe1c3-9d94-7061-9c69-0e332e80606b-0', tool_calls=[{'name': 'multiply', 'args': {'b': 7, 'a': 12}, 'id': '092d7731-e799-4e26-98a5-2782532f73d8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 103, 'total_tokens': 246})

In [7]:
msg.tool_calls

[{'name': 'multiply',
  'args': {'b': 7, 'a': 12},
  'id': '092d7731-e799-4e26-98a5-2782532f73d8',
  'type': 'tool_call'}]

In [8]:
# step 4: tool execution
result = multiply.invoke(llm_with_tools.invoke("can you multiply 3 with 10").tool_calls[0]['args'])
result

30

In [9]:
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

# step 1: define the tool
@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        return "Cannot divide by zero"
    return a / b


# Create toolkit
tools = [add, subtract, multiply, divide]

llm = ChatOllama(model="qwen3:8b")

#step2: bind the tool to the language model
llm_with_tools = llm.bind_tools(tools)

# -------------------------
# Check available tools
# -------------------------
for tool in tools:
    print("=" * 40)
    print(f"Tool: {tool.name}")
    print(f"Description: {tool.description}")
    print(f"Schema: {tool.args}")
    print(f"Schema JSON: {tool.args_schema.model_json_schema()}")
    print("=" * 40)

Tool: add
Description: Add two numbers.
Schema: {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
Schema JSON: {'description': 'Add two numbers.', 'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'add', 'type': 'object'}
Tool: subtract
Description: Subtract b from a.
Schema: {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
Schema JSON: {'description': 'Subtract b from a.', 'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'subtract', 'type': 'object'}
Tool: multiply
Description: Multiply two numbers.
Schema: {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
Schema JSON: {'description': 'Multiply two numbers.', 'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'multiply', 

In [10]:
# step3: tool calling
response = llm_with_tools.invoke("What is 25 multiplied by 12?")
response

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-08-08T14:25:53.8321264Z', 'done': True, 'done_reason': 'stop', 'total_duration': 15904180400, 'load_duration': 113146500, 'prompt_eval_count': 273, 'prompt_eval_duration': 86714000, 'eval_count': 194, 'eval_duration': 15681557000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019fe1c3-de87-7e31-b8fd-bd5f852d256e-0', tool_calls=[{'name': 'multiply', 'args': {'b': 12, 'a': 25}, 'id': '8698b56f-cbbf-4a91-941e-e05e1649502e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 273, 'output_tokens': 194, 'total_tokens': 467})

In [11]:
response.tool_calls

[{'name': 'multiply',
  'args': {'b': 12, 'a': 25},
  'id': '8698b56f-cbbf-4a91-941e-e05e1649502e',
  'type': 'tool_call'}]

In [12]:
print("LLM requested:")
print(response.tool_calls)

# Execute the requested tool
tool_call = response.tool_calls[0]

tool_name = tool_call["name"]
tool_args = tool_call["args"]

for tool in tools:
    if tool.name == tool_name:
        tool_result = tool.invoke(tool_args)
        break

print("\nTool result:")
print(tool_result)

LLM requested:
[{'name': 'multiply', 'args': {'b': 12, 'a': 25}, 'id': '8698b56f-cbbf-4a91-941e-e05e1649502e', 'type': 'tool_call'}]

Tool result:
300.0
